# Phase 6 — Suivi des portefeuilles

## Pourquoi cette phase est importante

Les trois portefeuilles sont déjà construits. Ils partagent les mêmes titres et les mêmes rendements. Cette phase regarde où va le poids, et à quel point il est concentré. Elle ne recalcule pas les poids.

Le suivi répond à trois questions.

- Quelle part du portefeuille est dans chaque secteur, chaque titre et chaque zone.
- Quelle part est en actions, en obligations et en ETF.
- Quelle part tiennent les dix premières lignes, et le risque de concentration qui en résulte.

## Les livrables

| Livrable | Fichier | Rôle |
|---|---|---|
| Documentation | `documentation/06_portfolio_monitoring.ipynb` | La méthode, les formules, les fichiers produits |
| Rapport mensuel | `reports/monitoring/06_rapport_mensuel.ipynb` | La situation des trois portefeuilles au dernier mois de la période |
| Dashboard | `notebooks/06_dashboard_monitoring.ipynb` et `reports/monitoring/dashboard.html` | Les graphiques des trois portefeuilles |

La vérification automatique est à part, dans `tests/monitoring/phase_06_validation.ipynb`. Elle ne remplace aucun livrable.

## Une situation, pas l'historique d'un mandat

Le rapport porte la date du dernier mois, lue dans `portfolio_metrics.csv`. Il décrit le portefeuille tenu ce mois-là. Les mois précédents ont leurs propres poids, fixés chacun avec les 60 mois déjà connus.

Le fichier des rendements mensuels applique ces derniers poids à tous les mois de la période. Le suivi ne le lit pas comme l'allocation qu'un mandat aurait eue au premier mois.


## Sources du registre

Le suivi part des portefeuilles déjà publiés. Il ne réestime pas les poids. Pour la situation, il relit la feuille de positions du benchmark. Pour l'historique, il relit les mêmes cours et le même SARON que la phase 5, afin de retrouver le rendement de chaque titre.

| Fichier lu | Origine | Rôle dans cette phase |
|---|---|---|
| `data/04_portfolios/portfolio_weights.csv` | sortie de la phase 5 | Les poids des trois stratégies |
| `data/04_portfolios/portfolio_metrics.csv` | sortie de la phase 5 | La période, le nombre de titres, le nombre effectif déjà publié |
| `data/04_portfolios/portfolio_monthly_returns.csv` | sortie de la phase 5 | Le rendement mensuel officiel. La somme des contributions doit le retrouver |
| `data/02_processed/market/market_prices_chf.csv` | `DS-003` et `DS-004` | Le rendement de chaque titre, pour la contribution des secteurs |
| `data/03_universe/eligible/chf_portfolio_universe.csv` | `DS-002` et `DS-004` | La même liste que la phase 5, pour reconstruire le panneau de rendements |
| `data/02_processed/macro/saron.csv` | `DS-005` | La même période que la phase 5 |
| `data/01_raw/benchmark/benchmark_holdings.xls` | `DS-000` | `Sector`, `Asset Class`, `Location` |

Le module ne contient pas l'identifiant `DS-000`. Il retrouve la feuille par le jeu qui produit `security_candidates.csv`, c'est-à-dire `DS-001`, dont l'entrée est le benchmark. Si un ticker y est répété, le calcul s'arrête.

`DS-006` n'est pas lu. L'ODD ne classe ni un secteur, ni une zone, ni une exposition.


## Entrées

Le module lit les poids publiés, la classification du benchmark, et les fichiers qui reconstruisent le rendement de chaque titre. Il ne réécrit ni les cours ni les poids.

| Rôle | Fichier | Colonnes utilisées |
|---|---|---|
| Poids | `data/04_portfolios/portfolio_weights.csv` | `strategy`, `yahoo_symbol`, `ticker`, `name`, `weight` |
| Période et contrôle | `data/04_portfolios/portfolio_metrics.csv` | `strategy`, `names`, `window_start`, `window_end`, `maximum_weight`, `effective_holdings` |
| Rendement officiel | `data/04_portfolios/portfolio_monthly_returns.csv` | `month`, `strategy`, `portfolio_return` |
| Cours de l'historique | `data/02_processed/market/market_prices_chf.csv` | `symbol`, `timestamp`, `close_chf` |
| Univers de l'historique | `data/03_universe/eligible/chf_portfolio_universe.csv` | `yahoo_symbol`, `chf_status` |
| Période SARON | `data/02_processed/macro/saron.csv` | `series_id`, `date`, `value`, `rate_currency` |
| Classification | `data/01_raw/benchmark/benchmark_holdings.xls`, jeu `DS-000` | `Ticker`, `Sector`, `Asset Class`, `Location` |

Aucun identifiant de jeu n'est écrit dans le module. Le fichier de positions est celui que le registre désigne déjà comme entrée des candidats. Si `Ticker` n'est pas unique dans cette feuille, le calcul s'arrête : un même ticker ne peut pas porter deux secteurs.

Un titre du portefeuille absent de cette feuille reste dans le portefeuille. Son secteur, son pays et sa classe d'actif deviennent `Non classé`. Le poids n'est pas retiré, et les totaux continuent de faire 1.

## Secteur, zone, exposition

Le secteur est le texte de la colonne `Sector`, sans traduction. Une valeur vide devient `Non classé`.

Le pays est le texte de `Location`. Une valeur vide, ou `--`, devient `Non classé`. La zone est un regroupement de ce pays :

| Zone | Pays publiés par la feuille de positions |
|---|---|
| Asie-Pacifique | Australia, China, Hong Kong, India, Japan, Korea (South), Malaysia, Singapore, Taiwan, Thailand |
| Europe | Austria, Belgium, Denmark, European Union, France, Germany, Italy, Norway, Portugal, Spain, Sweden, Switzerland, United Kingdom |
| Amériques | Brazil, Canada, Chile, Mexico, United States |
| Moyen-Orient et Afrique | Israel, Saudi Arabia, South Africa |

Un pays qui n'est pas dans cette table reste dans le portefeuille, dans la zone `Non classé`. Le journal technique le signale. Le calcul ne s'arrête pas, et le pays n'est pas rangé dans une zone au jugé.

L'exposition publiée est la devise de cotation. Le livre est composé d'actions cotées : les lignes obligations et ETF ne sont pas écrites, parce qu'elles seraient nulles par construction.

Une ligne dont le poids est inférieur ou égal à `1e-8` est de la poussière numérique de l'optimiseur, pas une position. Elle reste dans le fichier ligne à ligne et dans l'indice de Herfindahl. Elle sort des tableaux de secteur, de zone et d'exposition. Le seuil est très au-dessus de cette poussière, qui est de l'ordre de `1e-17`, et très au-dessous d'une ligne réellement publiée.


## Formules

Soit $w_i$ le poids publié du titre $i$, et $N$ le nombre de titres de l'univers commun. Ce nombre comprend les lignes que l'optimiseur a mises à zéro. Les trois portefeuilles ont le même $N$.

L'allocation d'un groupe $G$ (un secteur, une zone, ou une exposition) est

$$
w(G) = \sum_{i \in G} w_i
$$

Les groupes d'une même coupe se partagent le portefeuille. Leur somme vaut 1, à la poussière numérique près. Le module refuse de renormaliser un fichier de poids qui ne somme pas déjà à 1.

Les dix premières lignes sont les plus grands poids. À poids égal, l'ordre suit `yahoo_symbol`. S'il y a moins de dix lignes au-dessus du seuil, le top 10 est la somme de ces lignes.

$$
w_{(10)} = \sum_{k=1}^{\min(10, N_+)} w_{(k)}
$$

$N_+$ est le nombre de lignes au-dessus du seuil. Pour l'équipondéré, tous les poids sont égaux, donc $w_{(10)} = 10 / N$. Les dix noms affichés sont alors les dix premiers symboles, pas dix titres plus gros que les autres.

L'indice de Herfindahl utilise tous les poids publiés, y compris les zéros :

$$
H = \sum_{i=1}^{N} w_i^2
$$

Le nombre effectif de titres est

$$
N_{eff} = \frac{1}{H}
$$

Il vaut $N$ quand tous les poids sont égaux. Il baisse quand le poids se concentre. La phase 5 publie déjà $N_{eff}$. Cette phase le recalcule et s'arrête si les deux valeurs divergent.

Le multiple de concentration compare le portefeuille à l'équipondéré de même univers :

$$
C = N \, H
$$

$C = 1$ quand chaque titre pèse $1/N$. $C$ augmente quand le poids se concentre. Un multiple de 4 veut dire que le portefeuille est aussi concentré qu'un équipondéré de $N/4$ titres.


## Historique de performance

La situation décrit les poids du dernier mois. L'historique décrit ce que ces mêmes poids ont produit pendant la période.

Le poids d'un trimestre est fixé avec les soixante mois déjà connus. Il est tenu trois mois. Aucun émetteur ne dépasse 5 %, aucun secteur 20 %, aucune zone 50 %, aucune devise de cotation 40 %. La situation publiée est celle du mois du rapport.

Le rendement du titre $i$ au mois $t$ est $r_{i,t}$. Avec le poids cible $w_i$, sa contribution est

$$
c_{i,t} = w_i \, r_{i,t}
$$

La contribution d'un secteur est la somme des contributions de ses titres. Au mois $t$, la somme des secteurs est le rendement du portefeuille. La série cumulée additionne ces contributions. Elle n'est pas la valeur composée du portefeuille.

Un placement de 1 CHF part de 1 :

$$
W_t = \prod_{u \le t} (1 + R_u)
$$

Le drawdown est

$$
D_t = \frac{W_t}{\max_{u \le t} W_u} - 1
$$

$R_u$ est le rendement du portefeuille au mois $u$, avec le poids fixé à la fin du mois précédent. La performance cumulée enchaîne ces décisions. Elle n'est pas le résultat d'un poids unique gardé dix ans.

## Fichiers produits

Tous sont sous `data/05_monitoring/`. Chaque exécution les remplace.

| Fichier | Contenu |
|---|---|
| `allocation_by_holding.csv` | Une ligne par stratégie et par titre, avec secteur, pays, zone et exposition |
| `allocation_by_sector.csv` | Poids et nombre de lignes par secteur |
| `allocation_by_zone.csv` | Poids et nombre de lignes par zone |
| `exposure.csv` | Actions, obligations, ETF, puis toute autre classe présente |
| `concentration.csv` | Poids maximum, top 10, Herfindahl, nombre effectif, multiple |
| `top10_positions.csv` | Les dix premières lignes |
| `monthly_report.csv` | Une ligne par stratégie, datée du dernier mois de la période |
| `portfolio_path.csv` | Chaque mois : rendement, performance d'un placement de 1 CHF, drawdown |
| `sector_contribution.csv` | Contribution du secteur au rendement du mois, puis cumul |

La commande qui reconstruit ces fichiers seuls est

```
python -m src.portfolio_monitoring.build_monitoring
```

`python -m src.pipeline.refresh` le fait après la construction des portefeuilles, sur les poids qui viennent d'être estimés. La commande écrit aussi `reports/monitoring/dashboard.html`.

La cellule suivante lit le rapport déjà publié et affiche le mois, la période et l'exposition. Elle ne fixe aucune de ces valeurs.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import MONITORING_DATA

report = pd.read_csv(MONITORING_DATA / "monthly_report.csv")
row = report.iloc[0]
print(f"Mois du rapport : {row.report_month}")
print(f"Période : {row.window_start} à {row.window_end}")
print(f"Stratégies : {report['strategy'].nunique()}")
for item in report.itertuples(index=False):
    print(
        f"{item.strategy} : {int(item.invested_names)} lignes investies, "
        f"top 10 {item.top_10_weight:.1%}, "
        f"devise principale {item.largest_currency} {item.largest_currency_weight:.1%}."
    )


Mois du rapport : 2026-09
Période : 2016-10 à 2026-09
Stratégies : 3
equal_weight : 74 lignes investies, top 10 18.2%, devise principale USD 25.8%.
minimum_volatility : 38 lignes investies, top 10 46.7%, devise principale USD 22.3%.
maximum_sharpe : 35 lignes investies, top 10 48.0%, devise principale USD 39.4%.


# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas. Les formules restent dans la section qui les écrit.

**Allocation.** Somme des poids d'un groupe : un secteur, une zone ou une exposition.

**Benchmark.** Feuille de positions du proxy investissable, `DS-000`. Elle fournit le secteur, le pays et la classe d'actif. La référence stratégique est l'indice : elle n'est pas cette feuille.

**Contribution.** Poids d'un titre multiplié par son rendement du mois. La somme des contributions de secteur retrouve le rendement du portefeuille.

**Drawdown.** Baisse de la performance cumulée depuis le plus haut déjà atteint.

**Émetteur.** Société du titre. Le plafond de 5 % rappelé ici vient de la construction.

**ETF.** Fonds coté en bourse. Aucune ligne n'est publiée : le livre ne contient que des actions, et cette ligne serait nulle.

**Exposition.** Dans cette note, la devise de cotation.

**Herfindahl.** Somme des carrés des poids, y compris les poids nuls.

**Multiple de concentration.** Herfindahl multiplié par le nombre de titres. Il vaut 1 quand tous les poids sont égaux.

**Nombre effectif de titres.** Inverse du Herfindahl. Il baisse quand le poids se concentre.

**Non classé.** Secteur, pays, zone ou classe d'actif absent de la feuille de positions. Le poids reste dans le portefeuille.

**Obligation.** Classe d'actif absente du livre. La ligne n'est pas publiée.

**ODD.** Évaluation non lue dans cette phase. Elle ne classe ni un secteur, ni une zone, ni une exposition.

**Performance cumulée.** Valeur d'un placement de 1 CHF qui enchaîne les rendements mensuels.

**Poussière numérique.** Poids inférieur ou égal à 1e-8, résidu de l'optimiseur. Il reste dans le Herfindahl et sort des tableaux de secteur, de zone et d'exposition.

**SARON.** Fichier de taux relu pour retrouver la même période que la construction.

**Secteur.** Texte de la colonne `Sector`, sans traduction.

**Situation.** Portrait du portefeuille au dernier mois, pas l'allocation de chaque mois passé.

**Top 10.** Somme des dix plus grands poids au-dessus du seuil.

**Zone.** Regroupement du pays lu dans `Location` : Asie-Pacifique, Europe, Amériques, Moyen-Orient et Afrique.
